# 04 · Coding free text with the terminology mapper

Some concepts are fixed (body weight is always *Body weight*). Others depend on the note:
the name of a lab test, a condition, a drug. A column with a `map` source sends the
extracted text to a `TerminologyMapper` (notebook 01), which picks a standard concept.

The YAML says **what** to code; your Python code says **how**, by configuring the mapper.
This example uses the predefined `lab_results` definition (`measurement` table), whose
test name is coded like this:

```yaml
measurement_concept_id:
  from: text
  map: {}   # any standard concept of the column's domain (Measurement)
```

In [1]:
# Run from the folder where you ran `aatm init` (it holds .aatm/ and .env).
import os

if not os.path.isdir(".aatm") and os.path.isdir("../.aatm"):
    os.chdir("..")

## 1. Configure the mapper

Use the embedding model you chose in `aatm init` as the retriever. An LLM selector is
much better than the default closest match for short names such as "Hb".

In [2]:
from aatm.selectors import OpenAILLMSelector
from aatm.terminology_mapper import TerminologyMapper

mapper = TerminologyMapper(
    retriever="text-embedding-3-small",
    selector=OpenAILLMSelector("gpt-5-mini"),
)
mapper.retriever.top_k = 30  # candidates the selector sees

## 2. Run the pipeline with the mapper

In [3]:
from aatm.omop import ClinicalNote, OmopExtractionPipeline

notes = [
    ClinicalNote(
        text="Labs 03/15/2025: Hb 11.2 g/dL, creatinine 1.4 mg/dl, potassium 5.1 mmol/L, CRP < 5 mg/L.",
        person_id=10,
        note_date="2025-03-16",
    )
]

pipeline = OmopExtractionPipeline("lab_results", model_id="openai/gpt-5-mini", mapper=mapper)
tables = await pipeline(notes)

cols = ["measurement_source_value", "measurement_concept_id", "measurement_date", "operator_concept_id",
        "value_as_number", "unit_concept_id", "unit_source_value"]
tables["measurement"][cols]

,measurement_source_value,measurement_concept_id,measurement_date,operator_concept_id,value_as_number,unit_concept_id,unit_source_value
0,Hb,3000963,2025-03-15,<NA>,11.2,8713,g/dL
1,creatinine,3043098,2025-03-15,<NA>,1.4,8840,mg/dl
2,potassium,4245152,2025-03-15,<NA>,5.1,8753,mmol/L
3,CRP,4208414,2025-03-15,4171756,5.0,8751,mg/L


## 3. Review every coding decision

`mappings_dataframe()` has one row per coded text: `map` rows come from the mapper,
`lookup` rows from the UCUM unit lookup. Each distinct text is coded once per run.

In [4]:
tables.mappings_dataframe()[["row_id", "column", "text", "method", "concept_id", "concept_name", "vocabulary_id", "score"]]

,row_id,column,text,method,concept_id,concept_name,vocabulary_id,score
0,1,measurement_concept_id,Hb,map,3000963,Hemoglobin [Mass/volume] in Blood,LOINC,0.304537
1,1,unit_concept_id,g/dL,lookup,8713,gram per deciliter,UCUM,NaN
2,2,measurement_concept_id,creatinine,map,3043098,EPINEPHrine/Creatinine [Molar ratio] in Urine ...,LOINC,-0.081393
3,2,unit_concept_id,mg/dl,lookup,8840,milligram per deciliter,UCUM,NaN
4,3,measurement_concept_id,potassium,map,4245152,Potassium measurement,SNOMED,0.483654
5,3,unit_concept_id,mmol/L,lookup,8753,millimole per liter,UCUM,NaN
6,4,measurement_concept_id,CRP,map,4208414,C-reactive protein measurement,SNOMED,0.637604
7,4,unit_concept_id,mg/L,lookup,8751,milligram per liter,UCUM,NaN


Short names are ambiguous, so **always review this table**. In this run "Hb" and "CRP" are
right, "potassium" got the general SNOMED *Potassium measurement* (valid, but less
specific than a LOINC serum test), and "creatinine" was miscoded as a urine ratio with a
negative score.

## 4. Tune the `map` options

`map` accepts three options, set in YAML or in Python:

| Option | Effect |
|---|---|
| `domain` | search another domain than the column's default |
| `vocabularies` | only search these vocabularies, e.g. `[LOINC]`, if your data must use one |
| `min_score` | below this score, store `0` and report `low_confidence` |

Here we load the predefined file, change the options in Python, and **rebuild the table
from the same extraction**, so the notes are not sent to the LLM again.

In [5]:
from aatm.data_models.extractors import EntityDefinitions
from aatm.omop import MapSpec, OmopTableBuilder

definitions = EntityDefinitions.from_predefined("lab_results")
lab_test = next(e for e in definitions.entities if e.omop is not None)
lab_test.omop.fields["measurement_concept_id"][0].map = MapSpec(vocabularies=["LOINC"], min_score=0.35)

strict = OmopTableBuilder(definitions, mapper=mapper).build(notes, tables.extraction_results)
strict.mappings_dataframe()[["text", "concept_id", "concept_name", "vocabulary_id", "score", "stored"]]

,text,concept_id,concept_name,vocabulary_id,score,stored
0,Hb,3000963,Hemoglobin [Mass/volume] in Blood,LOINC,0.304537,0
1,g/dL,8713,gram per deciliter,UCUM,NaN,8713
2,creatinine,3043098,EPINEPHrine/Creatinine [Molar ratio] in Urine ...,LOINC,-0.081393,0
3,mg/dl,8840,milligram per deciliter,UCUM,NaN,8840
4,potassium,3023103,Potassium [Moles/volume] in Serum or Plasma,LOINC,0.406660,3023103
5,mmol/L,8753,millimole per liter,UCUM,NaN,8753
6,CRP,3020460,C reactive protein [Mass/volume] in Serum or P...,LOINC,0.408706,3020460
7,mg/L,8751,milligram per liter,UCUM,NaN,8751


In [6]:
strict.issues_dataframe()[["row_id", "column", "reason", "detail"]]

,row_id,column,reason,detail
0,1,measurement_concept_id,low_confidence,'Hb' -> 'Hemoglobin [Mass/volume] in Blood' (3...
1,2,measurement_concept_id,low_confidence,'creatinine' -> 'EPINEPHrine/Creatinine [Molar...


Restricting to LOINC made potassium and CRP more specific serum tests, and `min_score`
caught the wrong creatinine, but also dropped the correct "Hb" (0.30). Scores depend on
the retriever and selector, so look at real scores before choosing a `min_score`. Rows
below it are still returned, with `0` in the column and a `low_confidence` issue.

## Next

Several definitions at once, in the `observation` table: [05_observation.ipynb](05_observation.ipynb).